# CONDOR–FlexDC Generic Paired/Multi-Model Comparison — Full-Parity Workflow

This notebook is the version-neutral successor to the original V3-versus-V4 paired-comparison notebook. It preserves shared starts, shared simulator seeds, actual-point reuse, exact HTML, generated generalization workloads, interrupted-run recovery, Predict One, Optimize One, Rounds 1–4, explicit Round 5 handling, all-workload paired optimization, W&B, and complete packaging. Load any two or more complete generic training artifacts; model version names are labels rather than code paths.

## 0. Comparison, repository, artifact, fairness, W&B, and execution controls

In [ ]:
from pathlib import Path
import importlib.util, os, platform

IS_COLAB = importlib.util.find_spec("google.colab") is not None
RUN_ENV = "colab" if IS_COLAB else "local_pc"
WORKSPACE = Path("/content/workspace") if IS_COLAB else Path.cwd() / "flexdc_paired_workspace"
WORKSPACE.mkdir(parents=True, exist_ok=True)

COMDER_REPO_URL = "https://github.com/NetherMoon/CONDOR-FLEXDC.git"
FLEXDC_REPO_URL = "https://github.com/amenon871/FlexDC.git"
COMDER_BRANCH = FLEXDC_BRANCH = "main"
FORCE_FRESH_CLONE = False
UPDATE_EXISTING_REPOS = False
PRESERVE_LOCAL_REPO_CHANGES = True
CLONE_CONDOR_REPO = True
CLONE_FLEXDC_REPO = True
COMDER_ROOT_OVERRIDE = None
FLEXDC_ROOT_OVERRIDE = None
INSTALL_FLEXDC_REQUIREMENTS = False
INSTALL_FLEXDC_EDITABLE = True

PACKAGE_MODE = "path_zip" if IS_COLAB else "existing_dir"
PACKAGE_ZIP_INPUT = Path("/content/FlexDC_Generic_Full_Parity_Bundle.zip") if IS_COLAB else None
PACKAGE_DIR_INPUT = None
AUTO_UPLOAD_PACKAGE = True

# Two or more complete training artifact ZIPs/directories.
ARTIFACT_MODE = "multi_upload" if IS_COLAB else "existing"
MODEL_ARTIFACT_INPUTS = []
MODEL_LABEL_OVERRIDES = []
MODEL_CHECKPOINT_ROLES = []            # falls back to DEFAULT_CHECKPOINT_ROLE
DEFAULT_CHECKPOINT_ROLE = "best_feasibility"
PRIOR_RESULTS_INPUT = None
AUTO_UPLOAD_PRIOR_RESULTS = False
RESUME_EXISTING_RESULTS = True

COMPARISON_SEEDS = [30, 31, 32]
REQUIRE_ALL_SEEDS_PASS = True
VALIDATE_TOP_K_CANDIDATES = 3
CANDIDATE_DEDUP_DISTANCE = 0.03
VALIDATION_TIMEOUT_SECONDS = 1800
DRY_RUN_FLEXDC = False

# Exact shared P/R starts retained from the original paired notebook.
FATIH_P_START_VALUES = [0.6, 0.8, 1.0]
FATIH_R_START_VALUES = [0.2, 0.4, 0.6, 0.8]
EXPLICIT_START_PAIRS = [(p, r) for p in FATIH_P_START_VALUES for r in FATIH_R_START_VALUES]
ILLEGAL_START_POLICY = "project"      # project, skip, strict

TRACKING_MARGIN = 0.04
QOS_MARGIN = 0.01
R_OVER_P_MAX = 0.60
OPTIMIZATION_MODE = "margin_constrained"
OPTIMIZATION_ITERATIONS = 1500
OPTIMIZATION_LR = 0.03
OPTIMIZATION_MIN_LR = 5e-4
TRACKING_PENALTY = 2000.0
QOS_PENALTY = 2000.0
PENALTY_RAMP_FRACTION = 0.30
TOP_K = len(EXPLICIT_START_PAIRS)
LOG_EVERY = 25
RANDOM_SEED = 1700
TORCH_CPU_THREADS = 4

BASE_EXPERIMENT_RELATIVE = (
    "configs/experiment/new_iso/traditional_signal/generated_server_counts/"
    "exp_traditional_iso16_servers_1000.ini"
)
GRADIENT_CONFIG_RELATIVE = "configs/gradient_descent/gradient_descent_j2_pairwise_rsr.ini"
CLUSTER_CONFIG_RELATIVE = "configs/cluster/cluster.ini"

RUN_PREDICT_ONE = False
RUN_OPTIMIZE_ONE = False
RUN_ROUND_1 = False
RUN_ROUND_2 = False
RUN_ROUND_3 = False
RUN_ROUND_4 = False
RUN_ROUND_5 = False
RUN_ALL_WORKLOAD_PAIRED_OPTIMIZATION = False
ALL_WORKLOAD_RESUME = True
ALL_WORKLOAD_MAX_CASES = None
ALL_WORKLOAD_NAME_FILTER = None
ALL_WORKLOAD_CATEGORY_FILTER = None
ALL_WORKLOAD_STOP_ON_ERROR = False

PREDICT_ONE_CASE = {
    "id": "PREDICT_ONE_J2_CONTROL",
    "workload": "configs/workload/j2_pairwise/J2-IT-ResNetInf-GPT2Train.ini",
    "N": 1000, "U": 0.80, "duration": 3600,
    "P_ratio": 0.80, "R_ratio": 0.30,
}
OPTIMIZE_ONE_CASE = dict(PREDICT_ONE_CASE, id="OPTIMIZE_ONE_J2_CONTROL")

USE_WANDB = True
WANDB_MODE = "online"
WANDB_PROJECT = "flexdc-generic-paired-comparison"
WANDB_ENTITY = None
WANDB_RUN_ID_OVERRIDE = None
WANDB_FORCE_RELOGIN = False
WANDB_RUN_NAME = None
AUTO_DOWNLOAD_FINAL_ZIP = False
RUN_LABEL = "generic_paired_comparison"

print("Environment:", RUN_ENV)
print("Workspace:", WORKSPACE)
print("Models requested:", len(MODEL_ARTIFACT_INPUTS) or "upload")
print("Shared ratio starts:", EXPLICIT_START_PAIRS)
print("Shared seeds:", COMPARISON_SEEDS)


## 1. Verify dependencies without replacing Colab’s PyTorch stack

In [ ]:
import importlib.util, subprocess, sys
required = {
    "numpy":"numpy", "pandas":"pandas", "scipy":"scipy",
    "sklearn":"scikit-learn", "tqdm":"tqdm", "matplotlib":"matplotlib",
    "tabulate":"tabulate", "nbformat":"nbformat",
}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
if USE_WANDB and WANDB_MODE != "disabled" and importlib.util.find_spec("wandb") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wandb"], check=True)
import torch
print("Dependency check: PASS")
print("Torch:", torch.__version__, "CUDA:", torch.cuda.is_available())


## 2. Acquire the generic package and orchestration sources

In [ ]:
import shutil, zipfile, sys

def upload_one(description):
    if not IS_COLAB:
        raise FileNotFoundError(description)
    from google.colab import files
    print("Upload", description)
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise RuntimeError(list(uploaded))
    return (Path("/content") / next(iter(uploaded))).resolve()

if PACKAGE_MODE == "existing_dir":
    PACKAGE_ROOT = Path(PACKAGE_DIR_INPUT).expanduser().resolve() if PACKAGE_DIR_INPUT else Path.cwd()
else:
    package = Path(PACKAGE_ZIP_INPUT).expanduser().resolve() if PACKAGE_ZIP_INPUT else None
    if (package is None or not package.exists()) and AUTO_UPLOAD_PACKAGE:
        package = upload_one("FlexDC_Generic_Full_Parity_Bundle.zip")
    if package is None or not zipfile.is_zipfile(package):
        raise zipfile.BadZipFile(package)
    PACKAGE_ROOT = WORKSPACE / "generic_package"
    shutil.rmtree(PACKAGE_ROOT, ignore_errors=True)
    PACKAGE_ROOT.mkdir(parents=True)
    with zipfile.ZipFile(package) as archive:
        bad = archive.testzip()
        if bad:
            raise zipfile.BadZipFile(bad)
        archive.extractall(PACKAGE_ROOT)

hits = [PACKAGE_ROOT] if (PACKAGE_ROOT / "flexdc_generic_sources").exists() else [path.parent for path in PACKAGE_ROOT.rglob("flexdc_generic_sources")]
if not hits:
    raise FileNotFoundError("flexdc_generic_sources")
BUNDLE_ROOT = hits[0].resolve()
SOURCE_DIR = BUNDLE_ROOT / "flexdc_generic_sources"
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))
from flexdc_colab_orchestration import *
print("Bundle root:", BUNDLE_ROOT)


## 3. Clone/update both repositories, install FlexDC, and integrate generic sources

In [ ]:
COMDER_ROOT = Path(COMDER_ROOT_OVERRIDE).expanduser().resolve() if COMDER_ROOT_OVERRIDE else WORKSPACE / "comder-main"
FLEXDC_ROOT = Path(FLEXDC_ROOT_OVERRIDE).expanduser().resolve() if FLEXDC_ROOT_OVERRIDE else WORKSPACE / "FlexDC"
repo_states = []
if CLONE_CONDOR_REPO:
    repo_states.append(clone_or_update_repository(name="CONDOR-FLEXDC", url=COMDER_REPO_URL, destination=COMDER_ROOT, branch=COMDER_BRANCH, force_reclone=FORCE_FRESH_CLONE, update_existing=UPDATE_EXISTING_REPOS, preserve_local_changes=PRESERVE_LOCAL_REPO_CHANGES))
if CLONE_FLEXDC_REPO:
    repo_states.append(clone_or_update_repository(name="FlexDC", url=FLEXDC_REPO_URL, destination=FLEXDC_ROOT, branch=FLEXDC_BRANCH, force_reclone=FORCE_FRESH_CLONE, update_existing=UPDATE_EXISTING_REPOS, preserve_local_changes=PRESERVE_LOCAL_REPO_CHANGES))
if not FLEXDC_ROOT.exists():
    raise FileNotFoundError(f"FlexDC root is unavailable: {FLEXDC_ROOT}. Enable cloning or set FLEXDC_ROOT_OVERRIDE.")
REPO_MANIFEST = repository_manifest(repo_states, WORKSPACE / "repository_manifest.json")
if INSTALL_FLEXDC_REQUIREMENTS and (FLEXDC_ROOT / "requirements.txt").exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(FLEXDC_ROOT / "requirements.txt")], check=True)
if INSTALL_FLEXDC_EDITABLE and FLEXDC_ROOT.exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(FLEXDC_ROOT)], check=True)
GENERIC_SOURCE_INSTALL = install_generic_sources_to_condor(SOURCE_DIR, COMDER_ROOT / "am_flexdc" / "train")
for state in repo_states:
    print(state.to_dict())


## 4. Load all artifacts, reconstruct one shared runtime, restore prior outputs, and run tests

In [ ]:
import json, os, re, pandas as pd, numpy as np, torch
from IPython.display import HTML, display

ARTIFACT_ROOT = WORKSPACE / "paired_artifacts"
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
if ARTIFACT_MODE == "multi_upload":
    inputs = upload_files_direct(prompt="Upload two or more complete training artifact ZIPs.", multiple=True)
elif ARTIFACT_MODE == "upload_zip":
    inputs = upload_files_direct(prompt="Upload one complete training artifact ZIP.", multiple=False)
elif ARTIFACT_MODE in {"existing", "path_zip"}:
    inputs = [Path(path).expanduser().resolve() for path in MODEL_ARTIFACT_INPUTS]
else:
    raise ValueError(ARTIFACT_MODE)
if len(inputs) < 2:
    raise ValueError("The paired notebook requires at least two artifacts. Use the main inference notebook for one model.")

from flexdc_behavior_inference_utilities import *
from flexdc_inference_orchestration import *
from flexdc_behavior_evaluation import render_styled_table
from flexdc_presentation import build_report, write_report

MODEL_RUNTIMES = []
RUNTIME_INSTALLS = []
for index, source in enumerate(inputs):
    source = Path(source).expanduser().resolve()
    root = source if source.is_dir() else extract_zip_verified(source, ARTIFACT_ROOT / f"model_{index}", clean_destination=True, reject_exact_25_mib=False)[0]
    RUNTIME_INSTALLS.append(install_runtime_bundle(root, FLEXDC_ROOT))
    role = MODEL_CHECKPOINT_ROLES[index] if index < len(MODEL_CHECKPOINT_ROLES) else DEFAULT_CHECKPOINT_ROLE
    checkpoint = select_checkpoint(root, preferred_role=role)
    label = MODEL_LABEL_OVERRIDES[index] if index < len(MODEL_LABEL_OVERRIDES) else checkpoint.parent.name or f"Model_{index+1}"
    loaded = load_behavior_model(checkpoint, device_name="auto")
    MODEL_RUNTIMES.append({"label":safe_tag(label), "root":root, "checkpoint":checkpoint, "loaded":loaded, "epoch":int(loaded.checkpoint.get("epoch",-1)), "model_id":loaded.checkpoint.get("training_config",{}).get("model_id")})

torch.set_num_threads(max(1, int(TORCH_CPU_THREADS)))
for source in sorted(SOURCE_DIR.glob("*.py")):
    subprocess.run([sys.executable, "-m", "py_compile", str(source)], check=True)
env = os.environ.copy(); env["PYTHONPATH"] = str(SOURCE_DIR) + os.pathsep + env.get("PYTHONPATH", "")
for test in [SOURCE_DIR / "test_flexdc_behavior_training.py", SOURCE_DIR / "test_flexdc_profile_holdout.py", SOURCE_DIR / "test_flexdc_inference_orchestration.py"]:
    if test.exists():
        subprocess.run([sys.executable, str(test)], cwd=SOURCE_DIR, env=env, check=True)

OUTPUT_ROOT = WORKSPACE / "paired_results" / RUN_LABEL
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
PRIOR_RESULTS_ROOT = None
if PRIOR_RESULTS_INPUT:
    prior = Path(PRIOR_RESULTS_INPUT).expanduser().resolve()
    PRIOR_RESULTS_ROOT = extract_zip_verified(prior, WORKSPACE / "prior_paired_results", clean_destination=True, reject_exact_25_mib=False)[0] if prior.suffix.lower()==".zip" else prior
elif AUTO_UPLOAD_PRIOR_RESULTS:
    prior = upload_files_direct(prompt="Upload the previous paired-result ZIP.", multiple=False)[0]
    PRIOR_RESULTS_ROOT = extract_zip_verified(prior, WORKSPACE / "prior_paired_results", clean_destination=True, reject_exact_25_mib=False)[0]
RESTORED_PRIOR_FILES = restore_prior_outputs(PRIOR_RESULTS_ROOT, OUTPUT_ROOT) if RESUME_EXISTING_RESULTS else []

def show(frame, caption=None, precision=5, max_rows=500):
    if frame is None or len(frame)==0:
        print((caption or "Table") + ": no rows"); return
    render_styled_table(frame.head(max_rows), caption=caption, precision=precision)

show(pd.DataFrame([{k:(str(v) if k in {"root","checkpoint"} else v) for k,v in model.items() if k!="loaded"} for model in MODEL_RUNTIMES]), "Loaded paired models")
print("Recovered files:", len(RESTORED_PRIOR_FILES))


## 5. Start/resume the paired W&B run

In [ ]:
wandb_run = None
if USE_WANDB and WANDB_MODE != "disabled":
    import wandb, getpass
    os.environ["WANDB_MODE"] = WANDB_MODE
    if WANDB_MODE == "online":
        if WANDB_FORCE_RELOGIN:
            wandb.login(key=getpass.getpass("Paste W&B API key: "), relogin=True)
        else:
            try: wandb.login()
            except Exception: wandb.login(key=getpass.getpass("Paste W&B API key: "), relogin=True)
    wandb_run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name=WANDB_RUN_NAME or RUN_LABEL, mode=WANDB_MODE, id=WANDB_RUN_ID_OVERRIDE, resume="must" if WANDB_RUN_ID_OVERRIDE else None, config={"models":[m["label"] for m in MODEL_RUNTIMES], "seeds":COMPARISON_SEEDS, "ratio_starts":EXPLICIT_START_PAIRS})
    print("W&B:", wandb_run.url if getattr(wandb_run,"url",None) else wandb_run.id)
else:
    print("W&B disabled")


## 6. Exact explicit HTML presentation layer

In [ ]:
from flexdc_presentation import dataframe_html, multi_point_comparison_html
print("Explicit HTML presentation layer ready; Pandas Styler CSS will not be printed as raw text.")


## 7. Generate reusable workload and experiment configurations

In [ ]:
from configparser import ConfigParser
from copy import deepcopy
WORKLOAD_ROOT = FLEXDC_ROOT / "configs" / "workload"
EXPERIMENT_ROOT = FLEXDC_ROOT / "configs" / "experiment"
DEFAULT_SIMULATION_SEED = COMPARISON_SEEDS[0]

def render_html_table(frame, title, subtitle=""):
    html_text = build_report(title=title, subtitle=subtitle, tables={title:frame})
    display(HTML(html_text))

from configparser import ConfigParser
from copy import deepcopy
import pandas as pd

GENERALIZATION_WORKLOAD_DIR = WORKLOAD_ROOT / "generalization"
V4_WORKLOAD_DIR = WORKLOAD_ROOT / "v4_autogen"
AUTOGEN_EXPERIMENT_DIR = EXPERIMENT_ROOT / "reusable_inference_autogen"
for directory in [GENERALIZATION_WORKLOAD_DIR, V4_WORKLOAD_DIR, AUTOGEN_EXPERIMENT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

W1_SOURCE = WORKLOAD_ROOT / "W1-train-qos4444.ini"
W2_SOURCE = WORKLOAD_ROOT / "W2-short-qos5_4.5_4_3.5.ini"
for path in [W1_SOURCE, W2_SOURCE]:
    if not path.is_file():
        raise FileNotFoundError(path)


def read_ini(path):
    cfg = ConfigParser(interpolation=None)
    cfg.optionxform = str
    with Path(path).open("r", encoding="utf-8") as handle:
        cfg.read_file(handle)
    return cfg


def empty_ini():
    cfg = ConfigParser(interpolation=None)
    cfg.optionxform = str
    return cfg


def find_section(cfg, token, fallback_index):
    token = token.lower()
    aliases = [token]
    if token == "gpt2":
        aliases += ["gpt-2", "gpt"]
    for section in cfg.sections():
        if any(alias in section.lower() for alias in aliases):
            return section
    return cfg.sections()[fallback_index]


def unique_section_name(cfg, prefix, source_name):
    base = f"{prefix}__{source_name}"
    candidate = base
    counter = 2
    while cfg.has_section(candidate):
        candidate = f"{base}__{counter}"
        counter += 1
    return candidate


def copy_section(dst, dst_name, src, src_name, **overrides):
    if dst.has_section(dst_name):
        raise ValueError(f"Duplicate output section: {dst_name}")
    dst.add_section(dst_name)
    for key, value in src[src_name].items():
        dst.set(dst_name, key, str(value))
    # Materialize job_size so every generated section is explicit.
    if not dst.has_option(dst_name, "job_size"):
        dst.set(dst_name, "job_size", "1")
    for key, value in overrides.items():
        dst.set(dst_name, key, str(value))


def write_ini(cfg, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        cfg.write(handle, space_around_delimiters=True)
    return path


w1 = read_ini(W1_SOURCE)
w2 = read_ini(W2_SOURCE)
if len(w1.sections()) != 4 or len(w2.sections()) != 4:
    raise ValueError(f"Expected four W1 and four W2 sections; W1={w1.sections()}, W2={w2.sections()}")

profiles = {
    "TR": (w1, find_section(w1, "resnet", 0)),
    "TG": (w1, find_section(w1, "gpt2", 1)),
    "TL": (w1, find_section(w1, "llama", 2)),
    "TB": (w1, find_section(w1, "bloom", 3)),
    "IR": (w2, find_section(w2, "resnet", 0)),
    "IG": (w2, find_section(w2, "gpt2", 1)),
    "IL": (w2, find_section(w2, "llama", 2)),
    "IB": (w2, find_section(w2, "bloom", 3)),
}


def build_profile_workload(filename, entries, output_dir):
    """entries: [(profile_key, qos, output_prefix), ...] in exact job order."""
    cfg = empty_ini()
    for profile_key, qos, prefix in entries:
        source_cfg, source_section = profiles[profile_key]
        copy_section(
            cfg,
            unique_section_name(cfg, prefix, source_section),
            source_cfg,
            source_section,
            qos_constraint=float(qos),
        )
    return write_ini(cfg, output_dir / filename)


GENERATED_WORKLOADS = {}

# Round 2.
GENERATED_WORKLOADS["R2_T1"] = build_profile_workload(
    "GEN_R2_T1_W1_mixed_qos5_4p5_4_3p5.ini",
    [("TR", 5.0, "w1"), ("TG", 4.5, "w1"), ("TL", 4.0, "w1"), ("TB", 3.5, "w1")],
    GENERALIZATION_WORKLOAD_DIR,
)
GENERATED_WORKLOADS["R2_T2"] = build_profile_workload(
    "GEN_R2_T2_W2_all_qos4.ini",
    [("IR", 4.0, "w2"), ("IG", 4.0, "w2"), ("IL", 4.0, "w2"), ("IB", 4.0, "w2")],
    GENERALIZATION_WORKLOAD_DIR,
)
GENERATED_WORKLOADS["R2_T3"] = build_profile_workload(
    "GEN_R2_T3_W2_duplicate_resnet_all_qos4.ini",
    [("IR", 4.0, "w2"), ("IG", 4.0, "w2"), ("IL", 4.0, "w2"), ("IR", 4.0, "duplicate")],
    GENERALIZATION_WORKLOAD_DIR,
)
GENERATED_WORKLOADS["R2_T4"] = build_profile_workload(
    "GEN_R2_T4_W1_W2_hybrid_all_qos4.ini",
    [("TR", 4.0, "w1"), ("IG", 4.0, "w2"), ("TL", 4.0, "w1"), ("IB", 4.0, "w2")],
    GENERALIZATION_WORKLOAD_DIR,
)

# Round 4.
GENERATED_WORKLOADS["R4_T1_J3"] = build_profile_workload(
    "GEN_R4_T1_J3_W1_drop_bloom.ini",
    [("TR", 4.0, "w1"), ("TG", 4.0, "w1"), ("TL", 4.0, "w1")],
    GENERALIZATION_WORKLOAD_DIR,
)
GENERATED_WORKLOADS["R4_T2_J5"] = build_profile_workload(
    "GEN_R4_T2_J5_W1_plus_W2_GPT2.ini",
    [("TR", 4.0, "w1"), ("TG", 4.0, "w1"), ("TL", 4.0, "w1"), ("TB", 4.0, "w1"), ("IG", 4.0, "w2_added")],
    GENERALIZATION_WORKLOAD_DIR,
)
GENERATED_WORKLOADS["R4_T3_J6"] = build_profile_workload(
    "GEN_R4_T3_J6_W1_W2_three_profile_pairs.ini",
    [("TR", 4.0, "w1"), ("IR", 4.0, "w2"), ("TG", 4.0, "w1"), ("IG", 4.0, "w2"), ("TL", 4.0, "w1"), ("IL", 4.0, "w2")],
    GENERALIZATION_WORKLOAD_DIR,
)
GENERATED_WORKLOADS["R4_T4_J8_STRESS"] = build_profile_workload(
    "GEN_R4_T4_J8_W1_all_profiles_duplicated.ini",
    [("TR", 4.0, "copy1"), ("TG", 4.0, "copy1"), ("TL", 4.0, "copy1"), ("TB", 4.0, "copy1"),
     ("TR", 4.0, "copy2"), ("TG", 4.0, "copy2"), ("TL", 4.0, "copy2"), ("TB", 4.0, "copy2")],
    GENERALIZATION_WORKLOAD_DIR,
)

# Sweep V4 workload families — exact profile order and QoS vectors used in V4.
V4_DEFINITIONS = {
    "V4-H4-RG": [("IR", 3.0, "IR"), ("IG", 3.5, "IG"), ("TL", 4.5, "TL"), ("TB", 5.0, "TB")],
    "V4-H4-GL": [("TR", 4.0, "TR"), ("IG", 4.0, "IG"), ("IL", 3.5, "IL"), ("TB", 4.5, "TB")],
    "V4-H4-LB": [("TR", 4.5, "TR"), ("TG", 5.0, "TG"), ("IL", 3.0, "IL"), ("IB", 4.0, "IB")],
    "V4-H4-BR": [("IR", 3.5, "IR"), ("TG", 4.0, "TG"), ("TL", 5.0, "TL"), ("IB", 3.0, "IB")],
    "V4-J5-GPT2": [("TR", 5.0, "TR"), ("TG", 4.5, "TG"), ("TL", 4.0, "TL"), ("TB", 4.5, "TB"), ("IG", 3.5, "IG")],
    "V4-J5-Bloom": [("TR", 4.0, "TR"), ("TG", 5.0, "TG"), ("TL", 4.5, "TL"), ("TB", 5.0, "TB"), ("IB", 3.5, "IB")],
    "V4-J6-RGL": [("TR", 4.5, "TR"), ("IR", 3.0, "IR"), ("TG", 5.0, "TG"), ("IG", 3.5, "IG"), ("TL", 4.0, "TL"), ("IL", 3.0, "IL")],
    "V4-J8-ALL": [("TR", 5.0, "TR"), ("IR", 3.5, "IR"), ("TG", 4.0, "TG"), ("IG", 3.0, "IG"),
                   ("TL", 4.5, "TL"), ("IL", 3.5, "IL"), ("TB", 5.0, "TB"), ("IB", 4.0, "IB")],
}
for name, entries in V4_DEFINITIONS.items():
    GENERATED_WORKLOADS[name] = build_profile_workload(f"{name}.ini", entries, V4_WORKLOAD_DIR)

# Resolve the repository's standard experiment template.
BASE_EXPERIMENT = (
    EXPERIMENT_ROOT / "new_iso" / "traditional_signal" / "generated_server_counts" /
    "exp_traditional_iso16_servers_1000.ini"
)
if not BASE_EXPERIMENT.is_file():
    matches = list(EXPERIMENT_ROOT.rglob("exp_traditional_iso16_servers_1000.ini"))
    if len(matches) != 1:
        raise FileNotFoundError("Could not uniquely locate exp_traditional_iso16_servers_1000.ini")
    BASE_EXPERIMENT = matches[0]


def make_experiment_config(*, name, server_count, utilization, seed, duration_seconds=3600):
    cfg = read_ini(BASE_EXPERIMENT)
    if not cfg.has_section("system"):
        raise ValueError(f"Experiment template lacks [system]: {BASE_EXPERIMENT}")
    cfg.set("system", "server_count", str(int(server_count)))
    cfg.set("system", "utilization", f"{float(utilization):.10g}")
    cfg.set("system", "random_seed", str(int(seed)))
    cfg.set("system", "simulation_duration", str(int(duration_seconds)))
    return write_ini(cfg, AUTOGEN_EXPERIMENT_DIR / f"{name}.ini")

# Round 5 explicit contexts and durations.
ROUND5_EXPERIMENTS = {}
for context_id, utilization in [("N1000_U080", 0.80), ("N1000_U070", 0.70)]:
    for duration_id, seconds in [("1H", 3600), ("5H", 18000)]:
        key = f"{context_id}_{duration_id}"
        ROUND5_EXPERIMENTS[key] = make_experiment_config(
            name=f"GEN_R5_EXP_{context_id}_{duration_id}_{seconds}s",
            server_count=1000,
            utilization=utilization,
            seed=DEFAULT_SIMULATION_SEED,
            duration_seconds=seconds,
        )

config_rows = []
for name, path in GENERATED_WORKLOADS.items():
    config_rows.append({
        "ID": name,
        "File": str(path.relative_to(WORKLOAD_ROOT)),
        "Jobs": len(read_ini(path).sections()),
    })
GENERATED_WORKLOAD_TABLE = pd.DataFrame(config_rows).sort_values(["Jobs", "ID"])
render_html_table(
    GENERATED_WORKLOAD_TABLE,
    "Generated workload configurations",
    "Round 2, Round 4, and V4 workload files created from the repository profiles.",
)
print("Generated workload configs:", len(GENERATED_WORKLOAD_TABLE))
print("Generated Round 5 experiment configs:", len(ROUND5_EXPERIMENTS))

## 8. Shared starts, scenarios, prediction, optimization, and real-validation helpers

In [ ]:
BASE_EXPERIMENT = Path(BASE_EXPERIMENT)
GRADIENT_CONFIG = FLEXDC_ROOT / GRADIENT_CONFIG_RELATIVE
CLUSTER_CONFIG = FLEXDC_ROOT / CLUSTER_CONFIG_RELATIVE
for path in [BASE_EXPERIMENT, GRADIENT_CONFIG, CLUSTER_CONFIG]:
    if not path.exists(): raise FileNotFoundError(path)

def resolve_workload(spec):
    path = Path(spec)
    if path.is_absolute() and path.exists(): return path
    for candidate in [FLEXDC_ROOT / path, WORKLOAD_ROOT / path, WORKLOAD_ROOT / f"{path}.ini"]:
        if candidate.exists(): return candidate.resolve()
    matches = list(WORKLOAD_ROOT.rglob(path.name if path.suffix else f"{path.name}.ini"))
    if len(matches)!=1: raise FileNotFoundError(f"Could not uniquely resolve {spec}: {matches}")
    return matches[0].resolve()

def case_settings(job_count, seed=RANDOM_SEED):
    return OptimizationSettings(
        starts=max(len(EXPLICIT_START_PAIRS), len(EXPLICIT_START_PAIRS)),
        iterations=OPTIMIZATION_ITERATIONS, learning_rate=OPTIMIZATION_LR,
        minimum_learning_rate=OPTIMIZATION_MIN_LR, mode=OPTIMIZATION_MODE,
        tracking_penalty=TRACKING_PENALTY, qos_penalty=QOS_PENALTY,
        penalty_ramp_fraction=PENALTY_RAMP_FRACTION,
        top_k=min(TOP_K, len(EXPLICIT_START_PAIRS)),
        candidate_distance=CANDIDATE_DEDUP_DISTANCE, random_seed=seed,
        near_equal_start_fraction=0.0, high_p_low_r_start_fraction=0.0,
        enforce_flexdc_weight_bounds=True,
        weight_min_fraction_of_equal=0.60,
        weight_max_multiple_of_equal=1.80,
        r_over_p_max=R_OVER_P_MAX,
        explicit_ratio_starts=tuple(EXPLICIT_START_PAIRS),
        illegal_start_policy=ILLEGAL_START_POLICY,
        log_every=LOG_EVERY,
    )

def ratio_point(workload_path, experiment_path, p_ratio, r_ratio):
    workload = read_workload_config(workload_path)
    experiment = read_experiment_config(experiment_path)
    p_den, r_den = calculate_pr_denominators(workload, experiment)
    return p_ratio*p_den/(1000.0*experiment.server_count), r_ratio*r_den/(1000.0*experiment.server_count)

def scenario_from_spec(spec):
    workload_path = resolve_workload(spec["workload"])
    experiment_path = make_experiment_config(name=f"{safe_tag(spec['id'])}_N{spec['N']}_U{int(spec['U']*100):03d}_{spec.get('duration',3600)}s", server_count=spec["N"], utilization=spec["U"], seed=COMPARISON_SEEDS[0], duration_seconds=spec.get("duration",3600))
    if spec.get("mode","ratio") == "physical":
        pbar, reserve = float(spec["P"]), float(spec["R"])
    else:
        pbar, reserve = ratio_point(workload_path, experiment_path, float(spec.get("P",spec.get("P_ratio",1.0))), float(spec.get("R",spec.get("R_ratio",0.35))))
    jobs = read_workload_config(workload_path).job_count
    return ScenarioDefinition(case_id=spec["id"], workload_config=str(workload_path), experiment_config=str(experiment_path), server_count=spec["N"], utilization=spec["U"], initial_pbar=pbar, initial_r=reserve, initial_weights=tuple([1.0/jobs]*jobs), category=spec.get("category"))

def run_paired_cases(round_name, specs, *, run_enabled=True):
    if not run_enabled:
        print(round_name, "disabled"); return [], pd.DataFrame()
    cases = [scenario_from_spec(spec) for spec in specs]
    all_results=[]; summaries=[]
    for index, case in enumerate(cases):
        settings = case_settings(read_workload_config(case.workload_config).job_count, seed=RANDOM_SEED+index)
        # Optimized candidates differ by model, so each model's top-k is
        # validated separately. The fixed anchor is validated once per seed and
        # shared by all models below.
        results, summary = run_scenario_suite(
            model_runtimes=MODEL_RUNTIMES, cases=[case], output_root=OUTPUT_ROOT / safe_tag(round_name),
            settings=settings, tracking_margin=TRACKING_MARGIN, qos_margin=QOS_MARGIN,
            run_flexdc=True, validate_anchor=False, validate_top_k=True,
            simulator_seeds=COMPARISON_SEEDS, flexdc_root=FLEXDC_ROOT,
            gradient_config=GRADIENT_CONFIG, cluster_config=CLUSTER_CONFIG,
            validation_timeout_seconds=VALIDATION_TIMEOUT_SECONDS,
            resume=RESUME_EXISTING_RESULTS, dry_run_flexdc=DRY_RUN_FLEXDC,
        )
        shared_dir = OUTPUT_ROOT / safe_tag(round_name) / safe_tag(case.case_id) / "shared_anchor_actual"
        shared_dir.mkdir(parents=True, exist_ok=True)
        shared_rows = []
        canonical = MODEL_RUNTIMES[0]["loaded"]
        for shared_seed in COMPARISON_SEEDS:
            seeded = experiment_for_seed(
                case.experiment_config, shared_seed, shared_dir / "seed_experiments",
                utilization=case.utilization, server_count=case.server_count,
                simulation_duration=case.simulation_duration,
            )
            actual, jobs = run_flexdc_validation(
                python_executable=sys.executable, flexdc_root=FLEXDC_ROOT,
                gradient_config=GRADIENT_CONFIG, experiment_config=seeded,
                cluster_config=CLUSTER_CONFIG, workload_config=case.workload_config,
                output_label=f"{safe_tag(round_name)}_{safe_tag(case.case_id)}_shared_anchor_s{shared_seed}",
                pbar_kw_per_server=case.initial_pbar, r_kw_per_server=case.initial_r,
                weights=case.initial_weights, utilization=case.utilization,
                constants=canonical.constants, timeout_seconds=VALIDATION_TIMEOUT_SECONDS,
                dry_run=DRY_RUN_FLEXDC,
            )
            shared_rows.append({"Case_ID":case.case_id,"Seed":shared_seed,**actual})
        shared_actual = pd.DataFrame(shared_rows)
        dataframe_for_csv(shared_actual).to_csv(shared_dir / "shared_anchor_actual.csv", index=False)
        all_seed_pass = bool(shared_actual["Actual_Both_Pass"].astype(bool).all()) if len(shared_actual) and "Actual_Both_Pass" in shared_actual else False
        summary = summary.copy()
        summary["Shared_Anchor_All_Seeds_Pass"] = all_seed_pass
        summary["Shared_Anchor_Actual_Runs"] = len(shared_actual)
        all_results.extend(results); summaries.append(summary)
    table = pd.concat(summaries, ignore_index=True) if summaries else pd.DataFrame()
    show(table, f"{round_name} paired summary")
    if wandb_run is not None and len(table): wandb_run.log({f"{safe_tag(round_name)}/summary":wandb.Table(dataframe=table)})
    return all_results, table


## 8A. Restore and log already-completed paired outputs

In [ ]:
RECOVERED_COMPLETION = []
for path in sorted(OUTPUT_ROOT.rglob("case_summary.json")):
    try: RECOVERED_COMPLETION.append(json.loads(path.read_text()))
    except Exception: pass
print("Recovered completed case summaries:", len(RECOVERED_COMPLETION))
if wandb_run is not None:
    wandb_run.summary["recovered_case_summaries"] = len(RECOVERED_COMPLETION)


## 9. Predict one point with every model and one shared actual result

In [ ]:
PREDICT_ONE_MODELS = pd.DataFrame(); PREDICT_ONE_ACTUAL = pd.DataFrame(); PREDICT_ONE_COMPARISON = pd.DataFrame()
if RUN_PREDICT_ONE:
    case = scenario_from_spec({"id":PREDICT_ONE_CASE["id"], "workload":PREDICT_ONE_CASE["workload"], "N":PREDICT_ONE_CASE["N"], "U":PREDICT_ONE_CASE["U"], "duration":PREDICT_ONE_CASE["duration"], "mode":"ratio", "P":PREDICT_ONE_CASE["P_ratio"], "R":PREDICT_ONE_CASE["R_ratio"]})
    point = [{"Point":"Shared anchor", "pbar":case.initial_pbar, "r":case.initial_r, "weights":case.initial_weights}]
    PREDICT_ONE_MODELS = compare_models_on_points(model_runtimes=MODEL_RUNTIMES, workload_config=case.workload_config, experiment_config=case.experiment_config, points=point, server_count=case.server_count, utilization=case.utilization, tracking_margin=TRACKING_MARGIN, qos_margin=QOS_MARGIN)
    # Run the physical point once per seed and share the actual result with all models.
    rows=[]
    canonical = MODEL_RUNTIMES[0]["loaded"]
    for seed in COMPARISON_SEEDS:
        seeded = experiment_for_seed(case.experiment_config, seed, OUTPUT_ROOT/"predict_one"/"seed_experiments", utilization=case.utilization, server_count=case.server_count)
        actual,_ = run_flexdc_validation(python_executable=sys.executable, flexdc_root=FLEXDC_ROOT, gradient_config=GRADIENT_CONFIG, experiment_config=seeded, cluster_config=CLUSTER_CONFIG, workload_config=case.workload_config, output_label=f"paired_predict_one_s{seed}", pbar_kw_per_server=case.initial_pbar, r_kw_per_server=case.initial_r, weights=case.initial_weights, utilization=case.utilization, constants=canonical.constants, timeout_seconds=VALIDATION_TIMEOUT_SECONDS, dry_run=DRY_RUN_FLEXDC)
        rows.append({"Point":"Shared anchor","Seed":seed,**actual})
    PREDICT_ONE_ACTUAL = pd.DataFrame(rows)
    PREDICT_ONE_COMPARISON = PREDICT_ONE_MODELS.merge(PREDICT_ONE_ACTUAL.groupby("Point").first().reset_index(), on="Point", how="left")
    dataframe_for_csv(PREDICT_ONE_MODELS).to_csv(OUTPUT_ROOT/"predict_one_model_predictions.csv",index=False)
    dataframe_for_csv(PREDICT_ONE_ACTUAL).to_csv(OUTPUT_ROOT/"predict_one_shared_actual.csv",index=False)
    show(PREDICT_ONE_COMPARISON,"Predict One: all models, shared actual")


## 10. Optimize one workload with identical explicit starts and simulator seeds

In [ ]:
OPTIMIZE_ONE_RESULTS=[]; OPTIMIZE_ONE_SUMMARY=pd.DataFrame()
if RUN_OPTIMIZE_ONE:
    OPTIMIZE_ONE_RESULTS, OPTIMIZE_ONE_SUMMARY = run_paired_cases("Optimize One", [{"id":OPTIMIZE_ONE_CASE["id"],"workload":OPTIMIZE_ONE_CASE["workload"],"N":OPTIMIZE_ONE_CASE["N"],"U":OPTIMIZE_ONE_CASE["U"],"duration":OPTIMIZE_ONE_CASE["duration"],"mode":"ratio","P":OPTIMIZE_ONE_CASE["P_ratio"],"R":OPTIMIZE_ONE_CASE["R_ratio"]}], run_enabled=True)


## 11. Round 1 — unseen operating conditions

In [ ]:
ROUND1_TESTS = [
 {"id":"R1_T1_UNSEEN_UTILIZATION","workload":"W2-short-qos5555.ini","N":1000,"U":0.70,"duration":3600,"mode":"ratio","P":1.00,"R":0.35},
 {"id":"R1_T2_UNSEEN_SERVER_COUNT","workload":"W2-short-qos5555.ini","N":500,"U":0.80,"duration":3600,"mode":"ratio","P":1.00,"R":0.35},
 {"id":"R1_T3_COMBINED_INTERPOLATION","workload":"W2-short-qos5555.ini","N":500,"U":0.70,"duration":3600,"mode":"ratio","P":1.00,"R":0.35},
 {"id":"R1_T4_UTILIZATION_EXTRAPOLATION","workload":"W2-short-qos5555.ini","N":1000,"U":0.90,"duration":3600,"mode":"ratio","P":1.00,"R":0.35},
 {"id":"R1_T5_SERVER_COUNT_EXTRAPOLATION","workload":"W2-short-qos5555.ini","N":1500,"U":0.70,"duration":3600,"mode":"ratio","P":1.00,"R":0.35},
]
ROUND1_RESULTS, ROUND1_PAIRED_SUMMARY = run_paired_cases("Round 1", ROUND1_TESTS, run_enabled=RUN_ROUND_1)


## 12. Round 2 — new four-job mixes and QoS assignments

In [ ]:
ROUND2_TESTS = [
 {"id":"R2_T1_W1_MIXED_QOS","workload":GENERATED_WORKLOADS["R2_T1"],"N":1000,"U":0.80,"duration":3600,"mode":"ratio","P":1.00,"R":0.35},
 {"id":"R2_T2_W2_ALL_QOS4","workload":GENERATED_WORKLOADS["R2_T2"],"N":1000,"U":0.80,"duration":3600,"mode":"ratio","P":1.00,"R":0.35},
 {"id":"R2_T3_W2_DUPLICATE_RESNET","workload":GENERATED_WORKLOADS["R2_T3"],"N":1000,"U":0.80,"duration":3600,"mode":"ratio","P":1.00,"R":0.35},
 {"id":"R2_T4_W1_W2_HYBRID","workload":GENERATED_WORKLOADS["R2_T4"],"N":1000,"U":0.80,"duration":3600,"mode":"ratio","P":1.00,"R":0.35},
]
ROUND2_RESULTS, ROUND2_PAIRED_SUMMARY = run_paired_cases("Round 2", ROUND2_TESTS, run_enabled=RUN_ROUND_2)


## 13. Round 3 — existing unseen real workload

In [ ]:
ROUND3_TESTS = [
 {"id":"R3_MULTINODE_REFERENCE","workload":"W-train-multinode","N":1000,"U":0.70,"duration":3600,"mode":"physical","P":0.513669,"R":0.188335},
]
ROUND3_RESULTS, ROUND3_PAIRED_SUMMARY = run_paired_cases("Round 3", ROUND3_TESTS, run_enabled=RUN_ROUND_3)


## 14. Round 4 — variable job-count structural OOD

In [ ]:
ROUND4_TESTS = [
 {"id":"R4_T1_J3","workload":GENERATED_WORKLOADS["R4_T1_J3"],"N":1000,"U":0.80,"duration":3600,"mode":"ratio","P":1.00,"R":0.35},
 {"id":"R4_T2_J5","workload":GENERATED_WORKLOADS["R4_T2_J5"],"N":1000,"U":0.80,"duration":3600,"mode":"ratio","P":1.00,"R":0.35},
 {"id":"R4_T3_J6","workload":GENERATED_WORKLOADS["R4_T3_J6"],"N":1000,"U":0.80,"duration":3600,"mode":"ratio","P":1.00,"R":0.35},
 {"id":"R4_T4_J8_STRESS","workload":GENERATED_WORKLOADS["R4_T4_J8_STRESS"],"N":1000,"U":0.80,"duration":3600,"mode":"ratio","P":1.00,"R":0.35},
]
ROUND4_RESULTS, ROUND4_PAIRED_SUMMARY = run_paired_cases("Round 4", ROUND4_TESTS, run_enabled=RUN_ROUND_4)


## 15. Round 5 — duration sensitivity, explicitly separate from primary model score

In [ ]:
ROUND5_RESULTS=[]; ROUND5_PAIRED_SUMMARY=pd.DataFrame()
if RUN_ROUND_5:
    ROUND5_TESTS=[]
    for context_id, utilization in [("N1000_U080",0.80),("N1000_U070",0.70)]:
        for duration_id, seconds in [("1H",3600),("5H",18000)]:
            ROUND5_TESTS.append({"id":f"R5_{context_id}_{duration_id}","workload":"W2-short-qos5555.ini","N":1000,"U":utilization,"duration":seconds,"mode":"ratio","P":1.00,"R":0.35,"category":"Duration sensitivity"})
    ROUND5_RESULTS, ROUND5_PAIRED_SUMMARY = run_paired_cases("Round 5", ROUND5_TESTS, run_enabled=True)
    print("Round 5 is reported separately because simulation duration is not a current model input feature.")
else:
    print("Round 5 disabled. Enable only as a separate duration-sensitivity experiment; duration is not a model input.")


## 16. Paired optimization across all workloads, with resume and filters

In [ ]:
ALL_WORKLOAD_RESULTS=[]; ALL_WORKLOAD_PAIRED_SUMMARY=pd.DataFrame(); ALL_WORKLOAD_ERRORS=[]
if RUN_ALL_WORKLOAD_PAIRED_OPTIMIZATION:
    catalog=[]
    # Original and generated generalization workloads.
    for path in sorted(WORKLOAD_ROOT.rglob("*.ini")):
        category = "J2 pairwise" if "j2_pairwise" in str(path) else "Generated" if path.parent in {GENERALIZATION_WORKLOAD_DIR,V4_WORKLOAD_DIR} else "Original"
        catalog.append({"name":path.stem,"category":category,"path":path})
    selected=[]
    for item in catalog:
        if ALL_WORKLOAD_NAME_FILTER and not re.search(ALL_WORKLOAD_NAME_FILTER,item["name"],re.I): continue
        if ALL_WORKLOAD_CATEGORY_FILTER and ALL_WORKLOAD_CATEGORY_FILTER.lower() not in item["category"].lower(): continue
        selected.append(item)
    if ALL_WORKLOAD_MAX_CASES is not None: selected=selected[:int(ALL_WORKLOAD_MAX_CASES)]
    summaries=[]
    for index,item in enumerate(selected):
        spec={"id":f"ALL_{item['name']}_N1000_U080","workload":item["path"],"N":1000,"U":0.80,"duration":3600,"mode":"ratio","P":1.00,"R":0.35,"category":item["category"]}
        try:
            results,summary=run_paired_cases("All Workloads",[spec],run_enabled=True)
            ALL_WORKLOAD_RESULTS.extend(results); summaries.append(summary)
        except Exception as exc:
            ALL_WORKLOAD_ERRORS.append({"Case":spec["id"],"Error":repr(exc)})
            print("FAILED",spec["id"],repr(exc))
            if ALL_WORKLOAD_STOP_ON_ERROR: raise
    ALL_WORKLOAD_PAIRED_SUMMARY=pd.concat(summaries,ignore_index=True) if summaries else pd.DataFrame()
    show(ALL_WORKLOAD_PAIRED_SUMMARY,"Paired all-workload summary")
    if ALL_WORKLOAD_ERRORS: show(pd.DataFrame(ALL_WORKLOAD_ERRORS),"All-workload errors")
else:
    print("All-workload paired optimization disabled.")


## 17. Presentation-ready paired conclusions, W&B logging, packaging, and download

In [ ]:
summary_tables={
 "Predict One — model predictions":PREDICT_ONE_MODELS,
 "Predict One — shared actual":PREDICT_ONE_ACTUAL,
 "Optimize One":OPTIMIZE_ONE_SUMMARY,
 "Round 1":ROUND1_PAIRED_SUMMARY,
 "Round 2":ROUND2_PAIRED_SUMMARY,
 "Round 3":ROUND3_PAIRED_SUMMARY,
 "Round 4":ROUND4_PAIRED_SUMMARY,
 "Round 5 — separate":ROUND5_PAIRED_SUMMARY,
 "All workloads":ALL_WORKLOAD_PAIRED_SUMMARY,
}
report_html=build_report(title="CONDOR–FlexDC generic paired comparison",subtitle="All models use identical explicit ratio starts, optimizer settings, FlexDC seeds, and candidate-validation rules.",tables={name:frame for name,frame in summary_tables.items() if frame is not None and len(frame)},comparisons=[("Predict One models on the same point",PREDICT_ONE_MODELS)] if len(PREDICT_ONE_MODELS) else [],notes=["Fixed-point actual simulator results are generated once per seed and shared by all models.","Optimized candidates differ by model and are therefore validated separately using the same seed set."])
REPORT_HTML=write_report(OUTPUT_ROOT/"paired_presentation_ready_results.html",report_html)
if IS_COLAB: display(HTML(report_html))

if wandb_run is not None:
    for name,frame in summary_tables.items():
        if frame is not None and len(frame): wandb_run.log({safe_tag(name):wandb.Table(dataframe=frame)})
    wandb_run.summary["models"]=[model["label"] for model in MODEL_RUNTIMES]
    wandb_run.summary["shared_seeds"]=COMPARISON_SEEDS
    wandb_run.summary["explicit_ratio_starts"]=EXPLICIT_START_PAIRS
    wandb_run.summary["recovered_files"]=len(RESTORED_PRIOR_FILES)

RUN_MANIFEST=write_json(OUTPUT_ROOT/"paired_run_manifest.json",{
 "models":[{"label":model["label"],"checkpoint":str(model["checkpoint"]),"epoch":model["epoch"],"model_id":model.get("model_id")} for model in MODEL_RUNTIMES],
 "repositories":[state.to_dict() for state in repo_states],"runtime_installs":RUNTIME_INSTALLS,
 "seeds":COMPARISON_SEEDS,"explicit_ratio_starts":EXPLICIT_START_PAIRS,
 "flags":{"predict_one":RUN_PREDICT_ONE,"optimize_one":RUN_OPTIMIZE_ONE,"round1":RUN_ROUND_1,"round2":RUN_ROUND_2,"round3":RUN_ROUND_3,"round4":RUN_ROUND_4,"round5":RUN_ROUND_5,"all_workloads":RUN_ALL_WORKLOAD_PAIRED_OPTIMIZATION},
})
FINAL_ZIP=package_paths(OUTPUT_ROOT/f"{RUN_LABEL}_complete_outputs.zip",[OUTPUT_ROOT,SOURCE_DIR,REPO_MANIFEST,GENERALIZATION_WORKLOAD_DIR,V4_WORKLOAD_DIR,AUTOGEN_EXPERIMENT_DIR],root=WORKSPACE)
print("Package:",FINAL_ZIP); print("Size MB:",FINAL_ZIP.stat().st_size/1024**2); print("SHA-256:",sha256_file(FINAL_ZIP))
if wandb_run is not None:
    artifact=wandb.Artifact(name=RUN_LABEL,type="inference-comparison-results"); artifact.add_file(str(FINAL_ZIP)); wandb_run.log_artifact(artifact); wandb_run.finish()
download_if_colab(FINAL_ZIP,enabled=AUTO_DOWNLOAD_FINAL_ZIP)
